# Task 2: Exploratory Data Analysis & Preprocessing

## Objective
This notebook covers:
1. Comprehensive exploratory data analysis (EDA) with visualizations
2. Handling missing values and outliers
3. Addressing class imbalance using SMOTE or class weights
4. Engineering at least 3 new features with domain knowledge
5. Justification for all preprocessing choices
6. Saving processed data for modeling

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## 2.1 Load Merged Dataset

In [ ]:
# Load the merged dataset from Task 1
df = pd.read_csv('../data/raw/bank_merged_full.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
display(df.head())

In [ ]:
# Dataset info
print("Dataset Information:")
print("="*80)
df.info()

In [ ]:
# Statistical summary for numeric columns
print("Numerical Features Summary:")
print("="*80)
display(df.describe())

In [ ]:
# Categorical features summary
print("Categorical Features Summary:")
print("="*80)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print(f"Unique values: {df[col].nunique()}")

## 2.2 Missing Values Analysis

In [ ]:
# Detailed missing values analysis
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

print("Missing Values Summary:")
print("="*80)
display(missing_data)

# Visualize missing values
if len(missing_data) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    missing_data.plot(x='Column', y='Missing_Percentage', kind='bar', ax=ax, color='coral')
    ax.set_title('Missing Values by Feature', fontsize=14, fontweight='bold')
    ax.set_xlabel('Feature', fontsize=12)
    ax.set_ylabel('Missing Percentage (%)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../reports/figures/02_missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()

### Missing Values Strategy

**Economic Features (emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed)**:
- Missing for ~52% of data (from bank-full.csv)
- Strategy: Create indicator variable + median imputation
- Rationale: Preserves information about which records lacked economic data

**day_of_week**:
- Missing for ~52% of data (from bank-full.csv)
- Strategy: Fill with 'unknown' category
- Rationale: Maintains categorical structure

**day**:
- Missing for ~48% of data (from bank-additional-full.csv)
- Strategy: Can be dropped or filled with median
- Rationale: day_of_week provides similar information

In [ ]:
# Handle missing values
df_clean = df.copy()

# Economic features: Create indicator variables + median imputation
economic_features = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
for feature in economic_features:
    # Create indicator: 1 if missing, 0 otherwise
    df_clean[f'{feature}_missing'] = df_clean[feature].isnull().astype(int)
    # Impute with median from available data
    median_value = df_clean[feature].median()
    df_clean[feature].fillna(median_value, inplace=True)
    print(f"{feature}: Imputed with median = {median_value:.3f}")

# day_of_week: Fill with 'unknown'
df_clean['day_of_week'].fillna('unknown', inplace=True)
print("\nday_of_week: Filled with 'unknown'")

# day: Fill with median (or we could drop it)
day_median = df_clean['day'].median()
df_clean['day'].fillna(day_median, inplace=True)
print(f"day: Imputed with median = {day_median}")

# Verify no missing values remain
print(f"\nRemaining missing values: {df_clean.isnull().sum().sum()}")
print("✓ All missing values handled!")

## 2.3 Target Variable Analysis

In [ ]:
# Target distribution
print("Target Variable (y) Distribution:")
print("="*80)
target_counts = df_clean['y'].value_counts()
target_pct = df_clean['y'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Count': target_counts,
    'Percentage': target_pct.round(2)
})
display(target_summary)

imbalance_ratio = target_counts['no'] / target_counts['yes']
print(f"\nClass Imbalance Ratio (no:yes): {imbalance_ratio:.2f}:1")
print(f"This is a highly imbalanced dataset requiring special handling!")

# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
target_counts.plot(kind='bar', ax=axes[0], color=['#ff7f0e', '#2ca02c'])
axes[0].set_title('Target Variable Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subscribed', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d')

# Pie chart
colors = ['#ff7f0e', '#2ca02c']
axes[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Target Variable Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/02_target_distribution_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.4 Numerical Features Analysis

In [ ]:
# Select numerical columns (exclude indicator variables for now)
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remove indicator variables and day (we have day_of_week)
numeric_cols = [col for col in numeric_cols if not col.endswith('_missing') and col not in ['day']]

print(f"Numerical features to analyze: {numeric_cols}")

# Distribution plots
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df_clean[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frequency', fontsize=10)
    
# Hide extra subplots
for idx in range(len(numeric_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/02_numeric_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots by target variable
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    df_clean.boxplot(column=col, by='y', ax=axes[idx])
    axes[idx].set_title(f'{col} by Target', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Subscribed', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)
    plt.sca(axes[idx])
    plt.xticks([1, 2], ['no', 'yes'])

# Hide extra subplots
for idx in range(len(numeric_cols), len(axes)):
    axes[idx].axis('off')

plt.suptitle('')  # Remove default suptitle
plt.tight_layout()
plt.savefig('../reports/figures/02_numeric_boxplots_by_target.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.5 Outlier Detection and Treatment

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(data, column, multiplier=1.5):
    """
    Detect outliers using the IQR method.
    
    Args:
        data: DataFrame
        column: Column name
        multiplier: IQR multiplier (default 1.5)
    
    Returns:
        Boolean mask of outliers
    """
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    outliers = (data[column] < lower_bound) | (data[column] > upper_bound)
    return outliers, lower_bound, upper_bound

# Analyze outliers for each numeric column
print("Outlier Analysis (IQR method with multiplier=1.5):")
print("="*80)

outlier_summary = []
for col in numeric_cols:
    outliers, lower, upper = detect_outliers_iqr(df_clean, col)
    n_outliers = outliers.sum()
    pct_outliers = (n_outliers / len(df_clean) * 100)
    
    outlier_summary.append({
        'Feature': col,
        'Outlier_Count': n_outliers,
        'Outlier_Percentage': f"{pct_outliers:.2f}%",
        'Lower_Bound': lower,
        'Upper_Bound': upper
    })

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

### Outlier Treatment Strategy

**Decision: Capping instead of removal**
- Rationale: Outliers may contain valuable information (e.g., high balance customers)
- Method: Cap values at 99th percentile to reduce extreme values
- Exception: 'duration' - keep as is (highly predictive)
- Exception: Economic features - already continuous and meaningful

In [ ]:
# Cap outliers at 99th percentile for selected features
df_clean_capped = df_clean.copy()

# Features to cap (exclude duration and economic features)
features_to_cap = ['age', 'balance', 'campaign', 'pdays', 'previous']
features_to_cap = [f for f in features_to_cap if f in df_clean_capped.columns]

print("Capping outliers at 99th percentile:")
print("="*80)
for col in features_to_cap:
    p99 = df_clean_capped[col].quantile(0.99)
    n_capped = (df_clean_capped[col] > p99).sum()
    df_clean_capped[col] = df_clean_capped[col].clip(upper=p99)
    print(f"{col}: Capped {n_capped} values at {p99:.2f}")

print("\n✓ Outlier treatment completed!")

## 2.6 Categorical Features Analysis

In [ ]:
# Analyze categorical features
categorical_cols = [col for col in df_clean_capped.select_dtypes(include=['object']).columns 
                    if col not in ['y', 'data_source']]

print(f"Categorical features: {categorical_cols}")

# Visualize top categories for each feature
n_cols = 3
n_rows = int(np.ceil(len(categorical_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    top_categories = df_clean_capped[col].value_counts().head(10)
    top_categories.plot(kind='barh', ax=axes[idx])
    axes[idx].set_title(f'{col} (Top 10)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Count', fontsize=10)
    axes[idx].set_ylabel('', fontsize=10)

# Hide extra subplots
for idx in range(len(categorical_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/02_categorical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze categorical features vs target
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    # Get top categories
    top_cats = df_clean_capped[col].value_counts().head(10).index
    temp_df = df_clean_capped[df_clean_capped[col].isin(top_cats)]
    
    # Calculate subscription rate
    sub_rate = temp_df.groupby(col)['y'].apply(lambda x: (x == 'yes').sum() / len(x) * 100)
    sub_rate = sub_rate.sort_values(ascending=False)
    
    sub_rate.plot(kind='barh', ax=axes[idx], color='steelblue')
    axes[idx].set_title(f'{col} Subscription Rate', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Subscription Rate (%)', fontsize=10)
    axes[idx].set_ylabel('', fontsize=10)

# Hide extra subplots
for idx in range(len(categorical_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/02_categorical_subscription_rates.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.7 Correlation Analysis

In [ ]:
# Compute correlation matrix for numeric features
numeric_features_for_corr = [col for col in df_clean_capped.select_dtypes(include=['int64', 'float64']).columns
                             if col not in ['day'] and not col.endswith('_missing')]

corr_matrix = df_clean_capped[numeric_features_for_corr].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numerical Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/figures/02_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Find highly correlated features
print("\nHighly Correlated Feature Pairs (|correlation| > 0.7):")
print("="*80)
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

if high_corr:
    high_corr_df = pd.DataFrame(high_corr).sort_values('Correlation', key=abs, ascending=False)
    display(high_corr_df)
else:
    print("No highly correlated feature pairs found.")

## 2.8 Feature Engineering

### Domain-Specific Features
Based on banking domain knowledge and literature review, we create at least 3 new features:

1. **contact_efficiency**: Previous success rate = previous successes / max(1, previous contacts)
   - Rationale: Customers who responded positively in past are more likely to subscribe

2. **campaign_intensity**: Number of contacts in current campaign / duration
   - Rationale: High contact intensity may indicate desperation or strong interest

3. **economic_stability_score**: Composite score from economic indicators
   - Rationale: Favorable economic conditions increase likelihood of term deposit

4. **customer_value_proxy**: Balance + (age * 100) - representing potential customer value
   - Rationale: Wealthier and older customers may be better targets

5. **pdays_recency**: Transform pdays (999 = never contacted) to recency score
   - Rationale: Recent contacts are more likely to convert

6. **has_previous_contact**: Binary indicator if customer was contacted before
   - Rationale: Previous contact history is informative

7. **month_success_rate**: Historical success rate for each month
   - Rationale: Some months may be better for marketing

In [ ]:
# Feature Engineering
df_engineered = df_clean_capped.copy()

print("Engineering new features:")
print("="*80)

# 1. Contact Efficiency (previous success rate)
# poutcome: 'success' means previous campaign was successful
df_engineered['previous_success'] = (df_engineered['poutcome'] == 'success').astype(int)
df_engineered['contact_efficiency'] = df_engineered['previous_success'] / (df_engineered['previous'] + 1)
print("✓ contact_efficiency: previous_success / (previous + 1)")

# 2. Campaign Intensity
df_engineered['campaign_intensity'] = df_engineered['campaign'] / (df_engineered['duration'] / 60 + 1)  # calls per minute
# Replace any infinite values with 0
df_engineered['campaign_intensity'] = df_engineered['campaign_intensity'].replace([np.inf, -np.inf], 0)
print("✓ campaign_intensity: campaign / (duration_minutes + 1)")

# 3. Economic Stability Score (normalized composite)
# Only calculate for records that have economic data
economic_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
# Normalize each economic indicator to 0-1 scale
for col in economic_cols:
    min_val = df_engineered[col].min()
    max_val = df_engineered[col].max()
    df_engineered[f'{col}_norm'] = (df_engineered[col] - min_val) / (max_val - min_val)

# Create composite score (higher is more stable)
# Note: Some indicators are negative when economy is bad, so we need to handle signs appropriately
df_engineered['economic_stability_score'] = (
    df_engineered['emp.var.rate_norm'] * 0.3 +  # employment variation
    df_engineered['cons.price.idx_norm'] * 0.2 +  # consumer price index
    df_engineered['cons.conf.idx_norm'] * 0.2 +  # consumer confidence
    (1 - df_engineered['euribor3m_norm']) * 0.2 +  # euribor (lower is better for deposits)
    df_engineered['nr.employed_norm'] * 0.1  # number employed
)
print("✓ economic_stability_score: weighted composite of economic indicators")

# 4. Customer Value Proxy
df_engineered['customer_value_proxy'] = df_engineered['balance'] + (df_engineered['age'] * 100)
print("✓ customer_value_proxy: balance + (age * 100)")

# 5. Pdays Recency Score
# 999 means never contacted, transform to recency score (0-1, higher = more recent)
df_engineered['pdays_recency'] = np.where(
    df_engineered['pdays'] == 999,
    0,  # Never contacted = 0
    1 / (df_engineered['pdays'] + 1)  # More recent = higher score
)
print("✓ pdays_recency: transformed pdays to recency score (0-1)")

# 6. Has Previous Contact
df_engineered['has_previous_contact'] = (df_engineered['pdays'] != 999).astype(int)
print("✓ has_previous_contact: binary indicator (1 if previously contacted)")

# 7. Month Success Rate (encode based on historical data)
month_success_rate = df_engineered.groupby('month')['y'].apply(lambda x: (x == 'yes').sum() / len(x))
df_engineered['month_success_rate'] = df_engineered['month'].map(month_success_rate)
print("✓ month_success_rate: historical subscription rate for each month")

# 8. Additional: Age group
df_engineered['age_group'] = pd.cut(df_engineered['age'], 
                                     bins=[0, 30, 40, 50, 60, 100],
                                     labels=['<30', '30-40', '40-50', '50-60', '60+'])
print("✓ age_group: binned age into groups")

# 9. Duration in minutes (more interpretable)
df_engineered['duration_minutes'] = df_engineered['duration'] / 60
print("✓ duration_minutes: duration converted to minutes")

# Replace any infinite values in all numeric columns
numeric_features_eng = df_engineered.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_features_eng:
    df_engineered[col] = df_engineered[col].replace([np.inf, -np.inf], np.nan)
# Fill any NaN from inf replacement with median
for col in numeric_features_eng:
    if df_engineered[col].isnull().any():
        df_engineered[col].fillna(df_engineered[col].median(), inplace=True)
print("✓ Handled any infinite values in numeric features")

print(f"\n✓ Feature engineering completed! Added {9} new features.")
print(f"Original features: {df_clean_capped.shape[1]}")
print(f"After engineering: {df_engineered.shape[1]}")

In [ ]:
# Visualize engineered features
engineered_features = ['contact_efficiency', 'campaign_intensity', 'economic_stability_score', 
                       'customer_value_proxy', 'pdays_recency', 'month_success_rate']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(engineered_features):
    axes[idx].hist(df_engineered[col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frequency', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/02_engineered_features.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze engineered features vs target
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(engineered_features):
    df_engineered.boxplot(column=col, by='y', ax=axes[idx])
    axes[idx].set_title(f'{col} by Target', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Subscribed', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)
    plt.sca(axes[idx])
    plt.xticks([1, 2], ['no', 'yes'])

plt.suptitle('')
plt.tight_layout()
plt.savefig('../reports/figures/02_engineered_features_by_target.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.9 Class Imbalance Analysis and Handling

### Current State
- Imbalance ratio: ~7.8:1 (no:yes)
- This severe imbalance can lead to models that:
  - Achieve high accuracy but poor recall on minority class
  - Fail to identify positive cases (customers who would subscribe)

### Strategy
We'll prepare data for two approaches:
1. **SMOTE (Synthetic Minority Over-sampling Technique)**: Create synthetic samples
2. **Class Weights**: Adjust model training to penalize minority class errors more

Both approaches will be tested during model training phase.

In [ ]:
# Encode target variable for SMOTE preparation
from sklearn.preprocessing import LabelEncoder

df_final = df_engineered.copy()
label_encoder = LabelEncoder()
df_final['y_encoded'] = label_encoder.fit_transform(df_final['y'])

print("Target encoding:")
print(f"  'no' -> {label_encoder.transform(['no'])[0]}")
print(f"  'yes' -> {label_encoder.transform(['yes'])[0]}")

# Calculate class weights for later use
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', 
                                      classes=np.unique(df_final['y_encoded']), 
                                      y=df_final['y_encoded'])
class_weight_dict = dict(zip(np.unique(df_final['y_encoded']), class_weights))

print(f"\nClass weights for balanced training:")
print(f"  Class 0 (no): {class_weight_dict[0]:.3f}")
print(f"  Class 1 (yes): {class_weight_dict[1]:.3f}")

# Note: SMOTE will be applied during model training after train-test split
# to prevent data leakage

## 2.10 Final Data Preparation

In [ ]:
# Drop temporary normalized economic features
cols_to_drop = [col for col in df_final.columns if col.endswith('_norm')]
cols_to_drop.extend(['previous_success', 'data_source'])  # Keep data_source for reference if needed

print(f"Dropping temporary columns: {cols_to_drop}")
df_final = df_final.drop(columns=cols_to_drop, errors='ignore')

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Features: {df_final.shape[1] - 2} (excluding 'y' and 'y_encoded')")

In [ ]:
# Save processed data
import os

os.makedirs('../data/interim', exist_ok=True)

output_path = '../data/interim/bank_processed.csv'
df_final.to_csv(output_path, index=False)
print(f"✓ Processed dataset saved to: {output_path}")
print(f"  Shape: {df_final.shape}")
print(f"  Size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

In [ ]:
# Save class weights for later use
import joblib

os.makedirs('../config', exist_ok=True)
joblib.dump(class_weight_dict, '../config/class_weights.pkl')
joblib.dump(label_encoder, '../config/label_encoder.pkl')
print("✓ Class weights and label encoder saved to config/")

## 2.11 Summary and Key Insights

### Data Preprocessing Summary
1. **Missing Values**: 
   - Handled ~52% missing economic features with indicator variables + median imputation
   - Filled categorical missing values appropriately

2. **Outlier Treatment**:
   - Used IQR method for detection
   - Applied capping at 99th percentile for extreme values
   - Preserved 'duration' as is (highly predictive)

3. **Feature Engineering**:
   - Created 9 new domain-specific features
   - Features capture customer behavior, economic context, and campaign efficiency

4. **Class Imbalance**:
   - Identified 7.8:1 imbalance ratio
   - Prepared class weights for model training
   - SMOTE will be applied during modeling

### Key Insights from EDA
1. **Duration** is highly correlated with success (but only known post-call)
2. **Previous campaign outcome** is a strong predictor
3. **Economic indicators** show distinct patterns between classes
4. **Contact timing** (month, day_of_week) matters significantly
5. **Customer characteristics** (age, job, marital status) influence subscription

### Next Steps (Task 3: Model Development)
1. Encode categorical variables
2. Scale numerical features
3. Split data (train/validation/test)
4. Apply SMOTE on training data only
5. Train multiple models with hyperparameter tuning
6. Track experiments with MLflow

In [ ]:
# Final dataset overview
print("="*80)
print("FINAL PROCESSED DATASET SUMMARY")
print("="*80)
print(f"\nTotal Records: {len(df_final):,}")
print(f"Total Features: {df_final.shape[1]}")
print(f"\nNumeric Features: {len(df_final.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"Categorical Features: {len(df_final.select_dtypes(include=['object']).columns)}")
print(f"\nTarget Variable: y (binary: yes/no)")
print(f"  - Positive class (yes): {(df_final['y'] == 'yes').sum():,} ({(df_final['y'] == 'yes').sum() / len(df_final) * 100:.2f}%)")
print(f"  - Negative class (no): {(df_final['y'] == 'no').sum():,} ({(df_final['y'] == 'no').sum() / len(df_final) * 100:.2f}%)")
print(f"\n✓ Data preprocessing and feature engineering completed successfully!")
print(f"✓ Ready for model development (Task 3)")